# 🧊 28 · Heterogeneity inside cells: tomoDRGN and sub-tilt series
### Every particle in a tomogram is photographed many times — let the network look at all of its views at once

**BiologyToResearch** · Part F — Conformational heterogeneity: Gaussian mixtures and neural fields · Module 28 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/28_tomodrgn_subtilts.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 3 min to run everything); a GPU is optional |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-learn, PyTorch |
| **Prerequisites** | [27 · cryoDRGN](27_cryodrgn_neural_fields.ipynb) (neural fields, VAE); [07 · Tomography](07_tomography_missing_wedge.ipynb) (tilt series, missing wedge, dose-symmetric schemes); [06 · Image formation](06_electron_image_formation.ipynb) (radiation damage) |
| **Estimated time** | 100 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Powell & Davis 2024 (tomoDRGN, *Nat Methods*) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments are small, controlled *toy versions* of published methods (500 particles × 7 tilts of 32 × 32 pixels,
> small networks, a minute or two of training). They illustrate the core ideas; they do **not** reproduce the paper's
> architecture or numbers. **All molecules and images are generated by code** (nothing is downloaded); we leave out the
> CTF and the crowded cellular surroundings of real sub-tilts.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a **sub-tilt series** is — the set of 2D tilt images of one particle re-extracted from a tilt series —
   and why tomoDRGN learns from it instead of from 3D subtomograms.
2. **Compute** the pose of a particle in each tilt image, $P_{i,k} = Q_k\,T_i$ (tilt × particle pose).
3. **Model** radiation damage with the critical-exposure curve of Grant & Grigorieff and **build** tomoDRGN's per-tilt,
   per-frequency loss weights (dose attenuation, cos(tilt), optimal-exposure mask).
4. **Build and train** a tilt-set encoder: a shared network per tilt image (encoder A), a pooling step across tilts, and
   a second network (encoder B) that outputs one latent point per particle.
5. **Measure** how the number of tilts seen per particle changes the quality of the latent space, and **show** why
   encoding each tilt image separately (as single-particle cryoDRGN would) gives inconsistent answers.
6. **Relate** the toy to the paper's in-cell findings on ribosomes in *Mycoplasma pneumoniae*.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | tools, PyTorch helpers from Module 27 | 5 min |
| 1 · From subtomograms to sub-tilts | why 2D tilts; poses $Q_kT_i$ | 10 min |
| 2 · Radiation damage and the loss weights | critical exposure, cos(tilt), optimal-exposure mask | 15 min |
| 3 · A simulated sub-tilt data set | 500 particles × 7 tilts, dose-symmetric | 10 min |
| 4 · The tilt-set encoder | encoder A → pooling → encoder B; training | 20 min |
| 5 · How many tilts does a particle need? | latent quality vs tilts; per-tilt inconsistency | 15 min |
| 6 · Biology in situ | what tomoDRGN found in cells | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

Purified ribosomes in a test tube are not the whole story: in a living bacterium they are bound to messenger RNA, to
each other in **polysomes**, to the membrane while they make membrane proteins, and to drugs. Cryo-ET sees them there
(Modules 07–24), and subtomogram averaging (Module 14) turns tens of thousands of them into one map. But which
ribosomes are in which **state**, and where? Barrett Powell and Joseph Davis carried cryoDRGN into cells with
**tomoDRGN** (2024). Their key observation: in a tomogram each particle is not one picture but a whole **tilt series**
of pictures — and the network should look at all of them together.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

Helpers copied from [`course_toolkit.py`](../course_toolkit.py): the test protein of Part F, `move_domain`,
`gmm_project` and `reconstruct_fourier` (Module 25), and rotations (Module 02).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def rot_y(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def flexible_protein(seed: int = 7):
    """A code-generated ~80 kDa protein for heterogeneity experiments (Modules 25–30).

    Domains 0 and 1 form a rigid body; domain 2 (the 'arm') sits on domain 1 and can swing about a hinge; domain 3 is
    a small partner subunit that may be bound or not. Returns (atoms, hinge (x, y, z) in Å); atoms['domain'] labels
    every atom. Coordinates are in Å, centred near the origin.
    """
    rng = np.random.default_rng(seed)
    specs = [((21.0, 17.5, 16.0), (-19.0, -7.5, 0.0)), ((19.0, 16.0, 17.5), (19.0, -7.5, 0.0)),
             ((17.5, 12.5, 14.0), (21.0, 33.0, 0.0)), ((12.5, 11.0, 11.0), (-21.0, 26.0, 2.5))]
    parts = []
    for d, (radii, centre) in enumerate(specs):
        part = _transform(helix_bundle(radii, rng, chain="ABCD"[d]), _rz(rng.uniform(0, 2 * np.pi)), centre)
        part["domain"] = np.full(len(part["xyz"]), d)
        parts.append(part)
    atoms = merge_atoms(parts)
    shift = atoms["xyz"][atoms["domain"] < 3].mean(0)
    atoms["xyz"] = (atoms["xyz"] - shift).astype(np.float32)
    return atoms, np.array([19.0, 14.0, 0.0]) - shift


def move_domain(atoms, domain: int, R: np.ndarray, pivot) -> dict:
    """Copy of `atoms` with every atom of `domain` rotated by R (acting on (x, y, z)) about `pivot` (Å)."""
    out = dict(atoms)
    xyz = atoms["xyz"].astype(np.float64).copy()
    sel = atoms["domain"] == domain
    xyz[sel] = (xyz[sel] - pivot) @ np.asarray(R, float).T + pivot
    out["xyz"] = xyz.astype(np.float32)
    return out


def gmm_project(centres, amps, sigmas, R, box: int, pixel: float = 1.0) -> np.ndarray:
    """Projection [y, x] along z of a Gaussian mixture rotated by R (acting on (x, y, z)); same units as `gmm_to_map`.

    A 3D Gaussian projects to a 2D Gaussian of the same width, so the image is Σ_j w_j · g_y,j ⊗ g_x,j — two small
    matrices instead of a 3D grid (the 'project x and y separately' trick).
    """
    p = (np.asarray(centres, np.float32) @ np.asarray(R, np.float32).T)
    s = np.broadcast_to(np.asarray(sigmas, np.float32), (len(p),))[:, None]
    g = ((np.arange(box) - (box - 1) / 2.0) * pixel).astype(np.float32)[None, :]
    gx = np.exp(-(g - p[:, 0, None]) ** 2 / (2 * s ** 2))
    gy = np.exp(-(g - p[:, 1, None]) ** 2 / (2 * s ** 2))
    w = np.asarray(amps, np.float32)[:, None] * np.float32(np.sqrt(2 * np.pi)) * s / np.float32(pixel)   # ∫ dz
    return ((gy * w).T @ gx).astype(np.float32)


def reconstruct_fourier(images: np.ndarray, rotations, wiener: float = 1.0, chunk: int = 64) -> np.ndarray:
    """3D map [z, y, x] from projection images [n, y, x] with known rotations (R acts on (x, y, z), as in
    `rotate_volume`; image = rotate_volume(vol, R).sum(axis=0)).

    Projection-slice theorem: each image's 2D Fourier transform is the central slice of the 3D transform at
    k₃ = Rᵀ(kx, ky, 0). Slices are inserted with trilinear weights into a 2× padded grid, divided by the summed
    weights (+ `wiener`, in units of one image), transformed back and corrected for the interpolation kernel.
    """
    images = np.asarray(images, np.float64)
    n, N, _ = images.shape
    M, pad = 2 * N, N // 2
    k = np.arange(M) - M // 2
    ky, kx = np.meshgrid(k, k, indexing="ij")
    keep = (kx ** 2 + ky ** 2).ravel() <= (M // 2 - 1) ** 2
    plane = np.stack([kx.ravel(), ky.ravel(), np.zeros(M * M)], 1)[keep]
    shift_phase = np.exp(-1j * np.pi * (kx + ky) / M).ravel()[keep]          # half-pixel: image centre → index M/2
    Fr, Fi, W = np.zeros(M ** 3), np.zeros(M ** 3), np.zeros(M ** 3)
    corners = [(dz, dy, dx) for dz in (0, 1) for dy in (0, 1) for dx in (0, 1)]
    for start in range(0, n, chunk):
        idx_all, val_all, w_all = [], [], []
        for img, R in zip(images[start:start + chunk], list(rotations)[start:start + chunk]):
            big = np.zeros((M, M)); big[pad:pad + N, pad:pad + N] = img
            Fimg = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(big))).ravel()[keep] * shift_phase
            q = plane @ np.asarray(R, float)                                    # rows: (Rᵀ k)ᵀ in (x, y, z)
            pz = q[:, ::-1] + M // 2                                            # (z, y, x) grid coordinates
            i0 = np.floor(pz).astype(int); f = pz - i0
            for dz, dy, dx in corners:
                ii = i0 + (dz, dy, dx)
                wt = (np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1])
                      * np.where(dx, f[:, 2], 1 - f[:, 2]))
                ok = np.all((ii >= 0) & (ii < M), 1)
                idx_all.append((ii[ok, 0] * M + ii[ok, 1]) * M + ii[ok, 2]); val_all.append(wt[ok] * Fimg[ok])
                w_all.append(wt[ok])
        idx, val, wts = np.concatenate(idx_all), np.concatenate(val_all), np.concatenate(w_all)
        Fr += np.bincount(idx, val.real, M ** 3); Fi += np.bincount(idx, val.imag, M ** 3); W += np.bincount(idx, wts, M ** 3)
    F3 = ((Fr + 1j * Fi) / (W + wiener)).reshape(M, M, M)
    kz3, ky3, kx3 = np.meshgrid(k, k, k, indexing="ij")
    F3 *= np.exp(1j * np.pi * (kx3 + ky3 + kz3) / M)                            # centre back to (N − 1)/2
    vol = np.real(np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(F3))))[pad:pad + N, pad:pad + N, pad:pad + N]
    x = (np.arange(N) - (N - 1) / 2.0) / M
    sinc2 = np.sinc(x) ** 2                                                     # trilinear-kernel correction
    return (vol / (sinc2[:, None, None] * sinc2[None, :, None] * sinc2[None, None, :])).astype(np.float32)
print("🧰 toolkit loaded")

### Neural-field helpers (from Module 27)

The Hartley transform with its half-pixel centring, the positional encoding and the coordinate decoder
$V(\mathbf k, \mathbf z)$ — unchanged from Module 27.

In [ ]:
# ── Hartley transforms, positional encoding and the field decoder (Module 27) ──
N, PX = 32, 5.0
KC = np.arange(N) - N // 2


def ht2(imgs: np.ndarray) -> np.ndarray:
    """Centred 2D Hartley transform of images (..., N, N), with the half-pixel centring correction."""
    F = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(imgs, axes=(-2, -1))), axes=(-2, -1))
    F = F * np.exp(-1j * np.pi * (KC[:, None] + KC[None, :]) / N)
    return (F.real - F.imag).astype(np.float32)


W3 = torch.tensor(2 * math.pi * (N / 2) ** (np.arange(8) / 7), dtype=torch.float32)


def encode_3d(k):
    """Geometric positional encoding of 3D frequency coordinates (..., 3) → (..., 48)."""
    a = (k[..., :, None] * W3).reshape(*k.shape[:-1], -1)
    return torch.cat([torch.sin(a), torch.cos(a)], -1)


class FieldDecoder(nn.Module):
    """V(k, z): Hartley value of the map at 3D frequency k for latent point z."""

    def __init__(self, z_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(48 + z_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, k, z):
        return self.net(torch.cat([encode_3d(k), z[:, None, :].expand(-1, k.shape[1], -1)], -1))[..., 0]


print("helpers ready")

---
## 1 · From subtomograms to sub-tilts

> 📍 **Where we are.** Module 27 learned a conformational landscape from single-particle images — one image per
> particle. In a tomogram (Module 07), each particle appears in every image of the tilt series. This module feeds
> all of those views to one encoder. Module 29 then uses per-particle motions to sharpen maps, again with tilts.

The classical cryo-ET route (Modules 12–14) cuts a small **3D subtomogram** around each particle out of the
reconstructed tomogram. tomoDRGN goes back one step: it **re-extracts** the particle's small 2D patch from every
**tilt image** — a **sub-tilt series**. Why?

- A subtomogram has already been through reconstruction: its missing wedge (Module 07) is baked in as artefacts, and
  per-tilt corrections (CTF, dose) are blurred together. Each 2D sub-tilt is a clean measurement with its own known
  tilt, defocus and dose.
- Encoding 3D volumes is expensive; 2D images are what the cryoDRGN machinery already handles.

🧮 **Poses.** A subtomogram-averaging run gives each particle a 3D pose $T_i$ in the tomogram. The tilt image $k$ was
taken with the stage tilted by $\alpha_k$ about the y axis (rotation $Q_k$). So the particle's pose in tilt image $k$ is

$$P_{i,k} = Q_k\,T_i,$$

*in words:* first place the particle in the tomogram, then tilt the whole specimen. All tilts of particle $i$ share the
**same conformation** — the same latent point $\mathbf z_i$ — because they are pictures of the same molecule.

---
## 2 · Radiation damage and the loss weights

Every image in a tilt series costs electrons, and electrons destroy the specimen — fine details first (Module 06). In a
**dose-symmetric** scheme (Module 07) the low tilts are recorded first, the high tilts last, on a specimen that has
already received much of the dose. Grant & Grigorieff (2015) measured how fast signal fades: at spatial frequency $k$
(in Å⁻¹, 300 kV) the **critical exposure** is

$$N_c(k) = 0.24499\,k^{-1.6649} + 2.8141\ \ \text{e}^-/\text{Å}^2,
  \qquad \text{amplitude} \propto \exp\!\Big(-\frac{N}{2N_c(k)}\Big),$$

*in words:* after an accumulated exposure $N$ the amplitude at frequency $k$ has decayed by $e^{-N/2N_c}$; high
frequencies (small $N_c$) die quickly, low frequencies survive many electrons. tomoDRGN weights its loss accordingly:

- **dose weight** $\exp(-N_k/2N_c)$ for every frequency of tilt $k$ (do not demand that the model match signal that the
  beam has already destroyed);
- **tilt weight** $\cos\alpha_k$ (at high tilt the beam crosses more ice, so images are noisier);
- a **mask**: frequencies whose accumulated exposure exceeds the "optimal exposure" $2.51284\,N_c(k)$ are dropped.

In [ ]:
# ── 2.1 Critical exposure and the tomoDRGN weights for a dose-symmetric tilt scheme ──
TILTS = np.arange(-45, 46, 15).astype(float)                       # 7 tilts: −45° … +45° in 15° steps
T = len(TILTS)
DOSE_PER_TILT = 3.0                                                # e⁻/Å² per image
order = np.argsort(np.abs(TILTS) + 0.1 * (TILTS < 0))              # dose-symmetric: 0, +15, −15, +30, −30, …
dose_before = np.empty(T); dose_before[order] = np.arange(T) * DOSE_PER_TILT
k_inv_A = np.linspace(0.005, 0.1, 200)                              # 1/Å (200 Å … 10 Å)


def critical_exposure(k):
    """Grant & Grigorieff (2015) critical exposure (e⁻/Å², 300 kV) at spatial frequency k (1/Å)."""
    return 0.24499 * np.maximum(k, 1e-3) ** (-1.6649) + 2.8141


for a, n in zip(TILTS, dose_before):
    print(f"tilt {a:+5.0f}°: recorded after {n:4.1f} e⁻/Å², tilt weight cos = {np.cos(np.radians(a)):.2f}")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for n in sorted(set(dose_before)):
    axes[0].plot(1 / k_inv_A, np.exp(-n / (2 * critical_exposure(k_inv_A))), label=f"{n:.0f} e⁻/Å² before")
axes[0].set_xscale("log"); axes[0].invert_xaxis(); axes[0].set_xlabel("resolution (Å)")
axes[0].set_ylabel("dose weight exp(−N/2N_c)"); axes[0].legend(fontsize=6); axes[0].set_title("signal left after N electrons")
axes[1].plot(1 / k_inv_A, critical_exposure(k_inv_A), label="critical exposure N_c")
axes[1].plot(1 / k_inv_A, 2.51284 * critical_exposure(k_inv_A), label="optimal exposure 2.51 N_c")
axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].invert_xaxis(); axes[1].set_xlabel("resolution (Å)")
axes[1].set_ylabel("e⁻/Å²"); axes[1].legend(fontsize=7); axes[1].set_title("beyond the orange line: masked")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The first image (0°) is recorded on an intact specimen; by the last (±45°) the specimen has
absorbed 18 e⁻/Å², which at 10 Å resolution already halves the signal. Real tilt series use 41 or more images and a total
of ~100–150 e⁻/Å², so the last tilts keep only low-resolution information — the weights tell the network exactly how
much to trust each frequency of each image.

---
## 3 · A simulated sub-tilt data set

We reuse the three assembly states and the swinging arm of Module 27, now in a "tomogram": each particle has a random
3D pose $T_i$ and is imaged at the 7 tilts above. Each tilt image is damaged according to its accumulated dose, and its
noise grows with ice thickness ($1/\sqrt{\cos\alpha}$); SNR 0.3 at 0°.

In [ ]:
# ── 3.1 Simulate 500 particles × 7 tilts ──
atoms, hinge = flexible_protein()
Z_ATOM = np.array([{"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0}.get(e, 6.0) for e in atoms["element"]], np.float32)
STATE_NAMES = ["core", "core + arm", "complete"]
N_PART, SNR0 = 500, 0.3


def conformation(theta_deg: float, state: int):
    """Atom coordinates and amplitudes for an assembly state (0, 1, 2) with the arm turned by theta (Module 27)."""
    moved = move_domain(atoms, 2, rot_z(np.radians(theta_deg)), hinge)
    d = atoms["domain"]
    present = (d <= 1) | ((d == 2) & (state >= 1)) | ((d == 3) & (state >= 2))
    return moved["xyz"], (Z_ATOM * present).astype(np.float32)


KY2, KX2 = np.meshgrid(KC, KC, indexing="ij")
NC_2D = critical_exposure(np.sqrt(KX2 ** 2 + KY2 ** 2) / (N * PX))          # N_c at every pixel frequency
state_true = rng.integers(0, 3, N_PART)
theta_true = rng.uniform(-30, 30, N_PART)
pose_T = [random_rotation(rng) for _ in range(N_PART)]
subtilts = np.empty((N_PART, T, N, N), np.float32)
poses = np.empty((N_PART, T, 3, 3), np.float32)
t0 = time.time()
for i in range(N_PART):
    xyz, amp = conformation(theta_true[i], state_true[i])
    for k in range(T):
        poses[i, k] = rot_y(np.radians(TILTS[k])) @ pose_T[i]                 # P_ik = Q_k T_i
        img = gmm_project(xyz, amp, 2.5, poses[i, k], N, PX)
        F = np.fft.fftshift(np.fft.fft2(img)) * np.exp(-dose_before[k] / (2 * NC_2D))   # radiation damage
        subtilts[i, k] = np.real(np.fft.ifft2(np.fft.ifftshift(F)))
noise_sd = np.sqrt(subtilts[:, T // 2].var() / SNR0)
thickness = 1 / np.sqrt(np.cos(np.radians(TILTS)))                          # noise grows with path length
subtilts += (rng.normal(0, 1, subtilts.shape) * noise_sd * thickness[None, :, None, None]).astype(np.float32)
print(f"{N_PART} particles × {T} tilts = {N_PART * T} sub-tilt images in {time.time() - t0:.1f} s")

fig, axes = plt.subplots(1, T, figsize=(11, 1.9))
for k in range(T):
    axes[k].imshow(subtilts[0, k]); axes[k].set_title(f"{TILTS[k]:+.0f}°", fontsize=8); axes[k].axis("off")
plt.suptitle(f"one particle's sub-tilt series ({STATE_NAMES[state_true[0]]})", fontsize=9, y=1.05)
plt.show()

---
## 4 · The tilt-set encoder

How should one network read *several* images of one particle? tomoDRGN splits the encoder in two:

1. **Encoder A** — the same small network applied to **each** tilt image separately, producing a feature vector per tilt.
2. **Pooling** — combine the per-tilt features into one vector. The paper's default concatenates them (in tilt order);
   the code also offers max, mean, median and a learned set encoder. We use the **mean**, which works for any number of
   tilts and in any order — handy in Section 5.
3. **Encoder B** — a second network turns the pooled vector into the mean and variance of $q(\mathbf z\mid\text{all tilts})$.

The decoder is cryoDRGN's: for every tilt $k$ it renders the slice at pose $P_{i,k}$ from the **same** $\mathbf z_i$.
The loss is the dose- and tilt-weighted squared error over all tilts, plus the β-weighted KL term of Module 27 — with
β divided by the number of tilts, because one latent point now explains several images (in tomoDRGN, β depends on the
box size, the number of tilts and the latent dimension). (As in
Module 27, our per-tilt encoder is a small CNN rather than the paper's fully connected layers.)

In [ ]:
# ── 4.1 Data in Hartley space, slice coordinates and loss weights ──
RMASK = 10                                                                    # keep |k| ≤ 10 of 16 → 16 Å
in_circle = (KX2 ** 2 + KY2 ** 2) <= RMASK ** 2
H_all = ht2(subtilts)[:, :, in_circle]
H_SCALE = H_all.std()
H_t = torch.tensor(H_all / H_SCALE)                                            # (N_PART, T, M)
LATTICE = torch.tensor(np.stack([KX2[in_circle], KY2[in_circle], np.zeros(in_circle.sum())], 1) / N, dtype=torch.float32)
P_t = torch.tensor(poses)
nc = NC_2D[in_circle]
weights = (np.cos(np.radians(TILTS))[:, None] * np.exp(-dose_before[:, None] / (2 * nc[None]))
           * (dose_before[:, None] <= 2.51284 * nc[None]))                    # (T, M): tilt × dose × mask
W_t = torch.tensor(weights, dtype=torch.float32)
X_t = torch.tensor(subtilts / subtilts.std())
print(f"{in_circle.sum()} frequencies per sub-tilt; fraction of (tilt, frequency) pairs kept by the exposure mask: "
      f"{(weights > 0).mean():.2f}")

(Our short series never reaches the optimal-exposure limit at these low resolutions, so the mask keeps everything; in a
41-tilt series of 100+ e⁻/Å² the high-resolution frequencies of the last tilts would be dropped. The dose and tilt
weights still matter.)

In [ ]:
# ── 4.2 Encoder A (per tilt), mean pooling, encoder B, and the training loop ──
Z_DIM, FEAT = 2, 32


class EncoderA(nn.Module):
    """Shared per-tilt CNN: image (1, N, N) → feature vector (FEAT)."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                 nn.Linear(32, FEAT), nn.ReLU())

    def forward(self, x):
        return self.net(x)


def encode_particles(enc_a, enc_b, x):
    """x: (b, t, N, N) sub-tilts → (mu, logvar) of one latent point per particle (mean pooling over tilts)."""
    b, t = x.shape[:2]
    pooled = enc_a(x.reshape(b * t, 1, N, N)).reshape(b, t, FEAT).mean(1)
    out = enc_b(pooled)
    return out[:, :Z_DIM], out[:, Z_DIM:]


def train_tomodrgn(n_tilts: int, epochs: int = 10):
    """Train on random subsets of n_tilts tilts per particle. Returns (encoder A, encoder B, decoder)."""
    torch.manual_seed(SEED)
    enc_a, enc_b = EncoderA(), nn.Sequential(nn.Linear(FEAT, 64), nn.ReLU(), nn.Linear(64, 2 * Z_DIM))
    dec = FieldDecoder(Z_DIM)
    opt = torch.optim.Adam(list(enc_a.parameters()) + list(enc_b.parameters()) + list(dec.parameters()), lr=1e-3)
    t0 = time.time()
    for ep in range(epochs):
        for idx in torch.randperm(N_PART).split(8):
            sel = torch.stack([torch.randperm(T)[:n_tilts] for _ in idx])          # which tilts each particle shows
            mu, logvar = encode_particles(enc_a, enc_b, X_t[idx[:, None], sel])
            z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
            coords = torch.einsum("mj,bji->bmi", LATTICE, P_t[idx[:, None], sel].reshape(-1, 3, 3))
            pred = dec(coords, z.repeat_interleave(n_tilts, 0))                    # same z for all tilts of a particle
            target = H_t[idx[:, None], sel].reshape(-1, LATTICE.shape[0])
            w = W_t[sel].reshape(-1, LATTICE.shape[0])
            mse = (w * (pred - target) ** 2).sum() / w.sum()
            kl = (0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum(1)).mean()
            loss = mse + (1 / Z_DIM) * kl / (LATTICE.shape[0] * n_tilts)        # β also scaled by the number of tilts
            opt.zero_grad(); loss.backward(); opt.step()
    print(f"trained with {n_tilts} tilt(s) per particle: {epochs} epochs in {time.time() - t0:.0f} s "
          f"(weighted MSE {mse.item():.3f}, KL {kl.item():.2f})")
    return enc_a, enc_b, dec


def state_accuracy(z: np.ndarray) -> float:
    """Best-matched accuracy of 3 k-means clusters against the hidden assembly states."""
    lab = KMeans(3, n_init=10, random_state=SEED).fit_predict(z)
    C = np.zeros((3, 3)); np.add.at(C, (lab, state_true), 1)
    r, c = linear_sum_assignment(-C)
    return C[r, c].sum() / len(z)


model_all = train_tomodrgn(n_tilts=T)

---
## 5 · How many tilts does a particle need?

### 5.1 Fewer views at encoding time

Because we pool by averaging, the trained encoder can read any number of tilts. Let us encode every particle from
1, 2, 4 or all 7 of its tilts (chosen at random) and measure how well 3 k-means clusters of the latent points match the
hidden assembly states.

In [ ]:
# ── 5.1 Latent quality vs the number of tilts used to encode each particle ──
enc_a, enc_b, dec_all = model_all
acc_vs_tilts, z_by_n = {}, {}
for n_t in [1, 2, 4, T]:
    g = torch.Generator().manual_seed(n_t)
    sel = torch.stack([torch.randperm(T, generator=g)[:n_t] for _ in range(N_PART)])
    with torch.no_grad():
        z_by_n[n_t] = encode_particles(enc_a, enc_b, X_t[torch.arange(N_PART)[:, None], sel])[0].numpy()
    acc_vs_tilts[n_t] = state_accuracy(z_by_n[n_t])
    print(f"encoded from {n_t} tilt(s): state accuracy {acc_vs_tilts[n_t]:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), gridspec_kw=dict(width_ratios=[1, 1, 1.1]))
for ax, n_t in zip(axes[:2], [1, T]):
    for s, col in zip(range(3), ["tab:blue", "tab:orange", "tab:green"]):
        ax.scatter(*z_by_n[n_t][state_true == s].T, s=6, color=col, alpha=0.6, label=STATE_NAMES[s])
    ax.set_title(f"latent means from {n_t} tilt(s)", fontsize=9); ax.set_xlabel("z₁"); ax.set_ylabel("z₂")
axes[0].legend(fontsize=7)
axes[2].plot(list(acc_vs_tilts), list(acc_vs_tilts.values()), "o-")
axes[2].axhline(1 / 3, color="gray", ls="--", lw=1); axes[2].text(1.1, 0.35, "chance", fontsize=8, color="gray")
axes[2].set_xlabel("tilts seen per particle"); axes[2].set_ylabel("state accuracy"); axes[2].set_ylim(0.2, 1.0)
axes[2].set_title("more views → clearer latent space", fontsize=9)
plt.tight_layout(); plt.show()
assert acc_vs_tilts[T] > acc_vs_tilts[1], "pooling several tilts should beat a single tilt"

🔍 **What just happened?** From a single noisy tilt image the encoder can only guess; every additional view of the same
particle adds evidence, and the three assembly states separate more and more clearly. This is the paper's central design
choice made visible. The authors also report that training on random subsets of tilts works: as few as 11 of 41 tilts
per particle still gave a good latent space, while a single tilt degraded it sharply.

### 5.2 Why not simply run cryoDRGN on every tilt image?

Single-particle cryoDRGN would treat the 7 tilt images of a particle as 7 unrelated particles, each with its own latent
point. Let us encode each tilt *alone* and ask: do the 7 answers for the same particle agree?

In [ ]:
# ── 5.2 One latent point per tilt image: do the tilts of a particle agree? ──
with torch.no_grad():
    z_each = np.stack([encode_particles(enc_a, enc_b, X_t[:, k:k + 1])[0].numpy() for k in range(T)], 1)   # (N, T, 2)
km = KMeans(3, n_init=10, random_state=SEED).fit(z_by_n[T])
labels_each = km.predict(z_each.reshape(-1, Z_DIM)).reshape(N_PART, T)
consistent = np.mean([len(set(row)) == 1 for row in labels_each])
within = np.linalg.norm(z_each - z_each.mean(1, keepdims=True), axis=2).mean()       # tilts of one particle
between = np.linalg.norm(z_by_n[T] - z_by_n[T].mean(0), axis=1).mean()                 # different particles
print(f"particles whose 7 single-tilt encodings all fall in the same cluster: {consistent:.0%}")
print(f"scatter of one particle's single-tilt latent points, relative to the scatter between particles: "
      f"{within / between:.2f}")

🔍 **What just happened?** Encoded one tilt at a time, the same physical particle is scattered over the latent space and
often lands in different clusters — as if it changed its composition between exposures. A set encoder removes this
contradiction by construction: **one particle, one conformation, one latent point**, supported by all of its views.

✏️ **Your turn.** Train a second model with `train_tomodrgn(n_tilts=1)` (single tilts during training, about four times
faster) and repeat 5.1 with it. Is a network *trained* on single tilts worse at reading 7 tilts than one trained on all 7?

### 5.3 Back to the data: one map per cluster

As in Module 26, the safest way to see what the clusters mean is to go back to the measurements: reconstruct each
cluster directly from **all sub-tilts of its particles** (with their poses $P_{i,k}$; `reconstruct_fourier`, Module 25).
No network is involved in this step.

In [ ]:
# ── 5.3 Reconstruct each latent cluster from its particles' sub-tilts ──
cluster_maps = []
for c in range(3):
    sel = km.labels_ == c
    vol = reconstruct_fourier(subtilts[sel].reshape(-1, N, N), poses[sel].reshape(-1, 3, 3))
    main = np.bincount(state_true[sel], minlength=3)
    cluster_maps.append((vol, sel.sum(), STATE_NAMES[np.argmax(main)], main.max() / sel.sum()))
cluster_maps.sort(key=lambda m: m[0][m[0] > 0].sum())                   # order by reconstructed mass
fig, axes = plt.subplots(1, 3, figsize=(8, 2.8))
for ax, (vol, n, name, purity) in zip(axes, cluster_maps):
    ax.imshow(lowpass(vol, 0.3).sum(0)); ax.axis("off")
    ax.set_title(f"{n} particles: {purity:.0%} '{name}'", fontsize=9)
plt.suptitle("one reconstruction per latent cluster, from the raw sub-tilts", fontsize=10)
plt.tight_layout(); plt.show()

🔍 The three clusters reconstruct into the core, the core with its arm, and the complete protein — the assembly states,
recovered from noisy, damaged tilt images without labels. (Each cluster is not perfectly pure, so each map carries a
little of its neighbours.) Our toy does not resolve the arm's swing: like Module 27's, its per-tilt encoder would need
far more particles to see a small motion through changing orientations.

---
## 6 · Biology in situ

What can such a model reveal inside a cell? Powell & Davis applied tomoDRGN to a public tilt-series data set of
*Mycoplasma pneumoniae* cells treated with the antibiotic chloramphenicol (EMPIAR-10499 — the same data set used by
several Xu-lab papers in Modules 16–32). The authors report that, starting from about 975,000 template-matching
candidates and ≈ 22,000 ribosomes after classical cleaning (subtomogram average at 8.1 Å):

- a **homogeneous** tomoDRGN model already reached about the Nyquist limit of the data (≈ 3.5 Å);
- a **heterogeneous** model flagged remaining junk and isolated 50S-only particles (leaving ≈ 21,000 particles), and
  resolved states differing in bound tRNA and elongation factor EF-Tu, and motions of the L1 stalk and L7/L12 stalk;
- a **membrane-associated** subset (n = 482) showed extra density on the far side of the membrane in about 80 % of
  particles, which the authors suggest may be the SecDF complex, and about 13 % of membrane-bound ribosomes decoded
  EF-Tu·tRNA;
- maps could be placed back into the tomograms to look at **polysome** arrangements — structure *and* location.

💡 This is the dream of visual proteomics (Module 00) taken one step further: not only *where* each molecule is, but
*which state* it is in, particle by particle.

### 📄 Paper connection — Powell & Davis 2024: tomoDRGN

**Paper:** Powell BM, Davis JH. *Learning structural heterogeneity from cryo-electron sub-tomograms with tomoDRGN.*
Nature Methods 21:1525–1536 (2024). [doi:10.1038/s41592-024-02210-z](https://doi.org/10.1038/s41592-024-02210-z)
(preprint [bioRxiv 10.1101/2023.05.31.542975](https://doi.org/10.1101/2023.05.31.542975))

- **Question it answers:** how to learn the structural heterogeneity of particles *in their cellular context* from
  cryo-ET data, where each particle is seen in many low-dose tilt images.
- **What the paper actually does:** the input of each particle is its set of 2D tilt images (re-extracted sub-tilts) with
  poses from subtomogram averaging; all tilts share one latent point. A per-tilt encoder A (on the Hartley transform) is
  followed by pooling across tilts (default: concatenation; the code also has max, mean, median and a set encoder) and an
  encoder B. The cryoDRGN-style decoder uses Gaussian Fourier features. The loss is the squared error weighted per tilt by
  cos(tilt) and per frequency by the dose attenuation $\exp(-0.5N/N_c(k))$ with $N_c(k)=(0.24499\,k^{-1.6649}+2.8141)$
  × a voltage scale, masking frequencies beyond the optimal exposure $2.51284\,N_c$; the KL term is weighted by β.
  Training: AdamW, learning rate 2e-4, 50 epochs, batch size 1 (per the paper).
- **Result reported:** on simulated 50S assembly states (41 tilts, ±60° in 3° steps, 5,000 particles per state) four
  clusters matched the truth almost one-to-one; random subsets of as few as 11 of 41 tilts still gave a good latent
  space, a single tilt did not; and the *M. pneumoniae* findings of Section 6.
- **What our toy keeps:** sub-tilt poses $Q_kT_i$, dose and tilt weighting with the exposure mask, encoder A → pooling →
  encoder B, one latent point per particle, generated maps, the tilts-per-particle experiment.
- **What our toy leaves out:** CTF per tilt, concatenation pooling, Gaussian Fourier features, the cellular background,
  GPU-scale data — and therefore the subtle motions (L1 stalk, tRNA states) the real method resolves.
- **Code:** <https://github.com/bpowell122/tomodrgn>.

---
## ✅ Checkpoint

1. What is a sub-tilt series, and why does tomoDRGN prefer it to a subtomogram?
   <details><summary>Answer</summary>The 2D patches of one particle cut from every tilt image. Each is a clean measurement with its own tilt, defocus and dose; a subtomogram has the missing wedge and blurred corrections baked in, and is more expensive to encode.</details>
2. A particle has pose T in the tomogram and tilt image k was recorded at stage tilt α_k about y. What is its pose in that image?
   <details><summary>Answer</summary>P = Q_k T with Q_k the rotation by α_k about y: place the particle, then tilt the specimen.</details>
3. Why weight the loss by exp(−N/2N_c(k)) and drop frequencies beyond 2.51 N_c?
   <details><summary>Answer</summary>Radiation damage has already erased that much of the signal at frequency k; demanding that the model match it would only fit noise.</details>
4. Why must all tilts of a particle share one latent point, and how does the architecture enforce it?
   <details><summary>Answer</summary>They are images of one molecule in one conformation. The set encoder pools all tilts into one z, and the decoder renders every tilt from that z.</details>
5. What happens to the latent space when each particle is encoded from a single tilt?
   <details><summary>Answer</summary>It becomes much noisier (lower accuracy), and different tilts of the same particle can land in different clusters.</details>

## 🧠 Questions & discussion

1. Mean pooling ignores *which* tilt a feature came from; concatenation keeps the order. When could the order matter?
2. In a cell, neighbouring molecules overlap a particle's sub-tilts differently at every tilt. How might that confuse the
   encoder, and what does tomoDRGN's ability to flag "junk" suggest?
3. The membrane-bound subset was only 482 ribosomes. What statistics would you want before believing the extra density?
4. Why might a homogeneous tomoDRGN map be better than the classical subtomogram average of the same particles?
5. Dose-symmetric collection gives the low tilts the best signal. How should a set encoder weigh high-tilt images?

## 🚀 Bonus — going further

- **Concatenation pooling.** Replace `.mean(1)` in `encode_particles` by concatenating the 7 tilt features in tilt order
  (encoder B then needs `7 * FEAT` inputs) and compare the state accuracy. What breaks in Section 5.1?
- **No dose weighting.** Set all `weights` to 1 and retrain: does the latent space change?
- **More tilts.** Use 13 tilts (±60° in 10° steps); keep the total dose fixed by lowering `DOSE_PER_TILT`.
- **Real software:** the tomoDRGN documentation (<https://github.com/bpowell122/tomodrgn>) has a tutorial on EMPIAR-10499.

## 📝 Lab note

```text
Module 28 — date:
Input: 500 simulated particles × 7 tilts (±45°, dose-symmetric, 3 e⁻/Å² per tilt), 3 assembly states + arm swing
Output: dose/tilt weights, tilt-set VAE, latent quality vs tilts per particle, per-tilt inconsistency, generated maps
Method: P_ik = Q_k T_i, critical exposure (Grant & Grigorieff), encoder A → mean pooling → encoder B, weighted MSE + β·KL
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Powell & Davis 2024 (tomoDRGN)
```

## 📚 References

1. Powell BM, Davis JH. Learning structural heterogeneity from cryo-electron sub-tomograms with tomoDRGN. *Nat Methods* 21:1525–1536 (2024). doi:10.1038/s41592-024-02210-z
2. Zhong ED, Bepler T, Berger B, Davis JH. CryoDRGN: reconstruction of heterogeneous cryo-EM structures using neural networks. *Nat Methods* 18:176–185 (2021). doi:10.1038/s41592-020-01049-4
3. Grant T, Grigorieff N. Measuring the optimal exposure for single particle cryo-EM using a 2.6 Å reconstruction of rotavirus VP6. *eLife* 4:e06980 (2015). doi:10.7554/eLife.06980
4. Hagen WJH, Wan W, Briggs JAG. Implementation of a cryo-electron tomography tilt-scheme optimized for high resolution subtomogram averaging. *J Struct Biol* 197:191–198 (2017). doi:10.1016/j.jsb.2016.06.007
5. Tegunov D, Xue L, Dienemann C, Cramer P, Mahamid J. Multi-particle cryo-EM refinement with M visualizes ribosome-antibiotic complex at 3.5 Å in cells. *Nat Methods* 18:186–193 (2021). doi:10.1038/s41592-020-01054-7 — the *M. pneumoniae* data set (EMPIAR-10499).
6. Zaheer M *et al.* Deep sets. *NeurIPS* (2017). arXiv:1703.06114

## ➡️ Next

**[29 · Sharpening the moving parts: GMM-based focused refinement](29_gmm_focused_refinement.ipynb)** — use each
particle's motion to *re-align* the moving domain and recover a sharp map of it, in single-particle data and in tomograms.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).

In [ ]:
# ── Total run time ──
print(f"Notebook finished in {time.time() - T_START:.0f} s")